# Low-Rank-MoE 무료 Colab 야간 실행
이 노트북은 UI에서 직접 시작합니다. 먼저 이 노트북과 로컬에서 만든 `payload.tar.gz`, `payload_manifest.json`, `longer_training_bootstrap.py`를 Drive의 `Low-Rank-MoE-overnight/input/`에 올리세요. 학습 전에 별도 GPU 리허설과 새 런타임 복구 검증이 필요합니다. B와 C는 다른 밤에 실행합니다.


In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib, os, shutil, subprocess, sys, time
os.environ.setdefault('OVERNIGHT_SESSION_STARTED', str(time.monotonic()))
drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/Low-Rank-MoE-overnight')
INPUT = BASE / 'input'
OUTPUT = BASE / 'verified-runs'
assert all((INPUT / name).is_file() for name in ('payload.tar.gz', 'payload_manifest.json', 'longer_training_bootstrap.py', 'overnight_free_colab.ipynb')), 'Drive input is incomplete'
OUTPUT.mkdir(parents=True, exist_ok=True)
os.environ['COLAB_NOTEBOOK_UI'] = '1'
os.environ['OVERNIGHT_NOTEBOOK_SHA256'] = hashlib.sha256((INPUT / 'overnight_free_colab.ipynb').read_bytes()).hexdigest()
print('Drive access and four input files verified')


In [ ]:
shutil.copyfile(INPUT / 'payload.tar.gz', '/content/longer_payload.tar.gz')
shutil.copyfile(INPUT / 'payload_manifest.json', '/content/longer_payload_manifest.json')
shutil.copyfile(INPUT / 'longer_training_bootstrap.py', '/content/longer_training_bootstrap.py')
subprocess.run([sys.executable, '/content/longer_training_bootstrap.py'], check=True)
ROOT = Path('/content/low_rank_moe_longer')
CONTROLLER = ROOT / 'scripts/overnight_colab.py'


## GPU 리허설
처음에는 아래 리허설 셀을 실행합니다. 512 step 동안 초기·256·512 step Drive 세대를 검증합니다. 완료 후 **새 Colab 런타임**을 연결하고 위 두 셀을 다시 실행한 다음 복구 검증 셀을 실행하세요. 이 절차는 실제 B 학습과 별개입니다.


In [ ]:
RUN_REHEARSAL = False  # 처음 리허설을 시작할 때만 True로 변경
if RUN_REHEARSAL:
    subprocess.run([sys.executable, str(CONTROLLER), '--mode', 'rehearse', '--stage', 'B', '--drive', str(OUTPUT)], check=True)


In [ ]:
VERIFY_REHEARSAL = False  # 새 런타임에서만 True로 변경
if VERIFY_REHEARSAL:
    subprocess.run([sys.executable, str(CONTROLLER), '--mode', 'verify-rehearsal', '--stage', 'B', '--drive', str(OUTPUT)], check=True)


## 야간 B 또는 C 실행
리허설 보고서와 공간·전송 시간으로 총량을 확정한 뒤 값을 입력하세요. B는 네 조건 768 step씩, C는 네 조건 2048 step씩입니다. C는 B 완료 후 다음 밤에 별도로 시작합니다. 기존 런타임이 사라졌다면 새 런타임에서 첫 두 셀을 다시 실행하고 `RESUME=True`로 설정합니다.


In [ ]:
STAGE = 'B'
TOTAL_MINUTES = None    # 리허설 결과를 검토해 확정한 총 한도
SESSION_MINUTES = None  # 이번 Colab 세션의 한도
RESUME = False
DRIVE_AVAILABLE_BYTES = None  # Drive 계정 UI에서 확인한 보수적 잔여 바이트
assert STAGE in ('B', 'C') and TOTAL_MINUTES is not None and SESSION_MINUTES is not None
assert isinstance(DRIVE_AVAILABLE_BYTES, int) and DRIVE_AVAILABLE_BYTES > 0
LIMITS = ['--stage', STAGE, '--drive', str(OUTPUT), '--total-seconds', str(TOTAL_MINUTES * 60), '--session-seconds', str(SESSION_MINUTES * 60), '--drive-available-bytes', str(DRIVE_AVAILABLE_BYTES)]
subprocess.run([sys.executable, str(CONTROLLER), '--mode', 'preflight', *LIMITS], check=True)


In [ ]:
START_TRAINING = False  # 사전 검사 출력과 한도를 확인한 뒤 True로 변경
if START_TRAINING:
    command = [sys.executable, str(CONTROLLER), '--mode', 'run', *LIMITS]
    if RESUME: command.append('--resume')
    subprocess.run(command, check=True)
